# Actividad 1 — Auditoría y preparación de un corpus

Valor: 8% de la materia.

Una organización ficticia recibe solicitudes de soporte en español. Antes de utilizar estos mensajes para desarrollar un clasificador, necesita un corpus confiable y documentado. Actúas como analista de datos: debes recibir el corpus, auditarlo, decidir qué conservar o modificar, preparar una versión reproducible y separar los datos para un uso posterior.

El producto de esta actividad es un corpus preparado y documentado. No se entrena ningún modelo ni se calculan predicciones o métricas de clasificación.

## Objetivo
Aplicar una auditoría a textos imperfectos, diseñar una preparación adecuada para español y justificar, con ejemplos verificables, qué información conservas, modificas o excluyes. Cada transformación importante debe conectar problema, decisión, implementación, justificación y comprobación.

## Lo que ya debes poder utilizar
De Clase 1: entrada y salida, corpus, etiquetas provisionales, contexto, ambigüedad y negación. De Clase 2: DataFrame, faltantes, duplicados, Unicode/NFC, UTF-8, normalización, regex básica, tokenización, privacidad, familias, particiones y reconstrucción desde un kernel nuevo.

Tecnologías: Python, pandas y spaCy; re y unicodedata forman parte de Python. Jupyter local o Google Colab, con CPU. Para tokenizar usamos spacy.blank("es"), ya explicado en la práctica de Clase 2: no descarga un modelo entrenado y no obtiene lemas. No se exige usar también NLTK ni repetir el trabajo con otra biblioteca.

Puedes consultar las presentaciones y notebooks de Clases 1 y 2 durante toda la actividad. Los pasos y operaciones de apoyo están indicados en cada sección; las decisiones sobre este corpus son tuyas.

## Datos y diccionario
Usa exclusivamente data/corpus_auditoria_v2.csv. Es un CSV en UTF-8 con textos sintéticos autorizados para docencia. Consulta también data/README.md.

| Campo | Significado y tratamiento |
| --- | --- |
| id | Identificador estable del registro; no renumerarlo. |
| text | Mensaje original; conservarlo como text_original. |
| label | Categoría provisional: acceso (cuenta, contraseña, autenticación) o pagos (cargos, facturas, recibos, reembolsos). Puede requerir revisión; no inventar etiquetas. |
| family | Procedencia común de variantes relacionadas; mantenerlas juntas en la partición. No es una etiqueta ni una pista para completarla. |

Convención del archivo: `__NULO__` significa dato no recibido; un campo vacío es una cadena vacía. Los espacios también deben inspeccionarse. La carga proporcionada usa `keep_default_na=False` y `na_values=["__NULO__"]` para distinguirlos. Reconocer un faltante no decide automáticamente qué hacer con su fila.

El corpus contiene casos para tomar decisiones; no se especifican sus ubicaciones. No visites los enlaces ni envíes mensajes a los correos: son ficticios. En este ejercicio académico puedes conservar el original en tu entrega, porque todos los datos son sintéticos. Para datos reales, preservar original exige acceso controlado; reemplazar correos no garantiza anonimización total.

## Preparación: elige un entorno
Colab (preferido): abre el notebook Alumno en Google Colab. Sube corpus_auditoria_v2.csv al panel Archivos. Ejecuta la preparación; instala solo las bibliotecas que falten. Al terminar descarga el notebook ejecutado y los archivos de entrega/ antes de cerrar la sesión.

In [1]:
from pathlib import Path
import sys, subprocess, importlib.util, platform, json, re, unicodedata, hashlib
from importlib.metadata import version

# Instala solamente las dos bibliotecas de la actividad si faltan.
faltan = []
if importlib.util.find_spec("pandas") is None:
    faltan.append("pandas==2.2.3")
if importlib.util.find_spec("spacy") is None:
    # Versiones de apoyo compatibles; no son nuevas herramientas de la actividad.
    faltan.extend(["spacy==3.8.7", "typer==0.16.0", "typer-slim==0.16.0", "click==8.1.8"])
elif importlib.util.find_spec("click") is None:
    faltan.extend(["typer==0.16.0", "typer-slim==0.16.0", "click==8.1.8"])
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", *faltan], check=True)
import pandas as pd
import spacy
print({"Python": platform.python_version(), "pandas": pd.__version__,
       "spaCy": spacy.__version__})


{'Python': '3.14.4', 'pandas': '3.0.5', 'spaCy': '3.8.16'}


In [2]:
# Local: encuentra data/ junto al notebook o en una carpeta superior.
# Colab: acepta también el CSV subido al panel Archivos, en el directorio actual.
nombre = "corpus_auditoria_v2.csv"
candidatos = [Path.cwd() / nombre]
candidatos += [base / "data" / nombre for base in [Path.cwd(), *Path.cwd().parents]]
ARCHIVO = next((p for p in candidatos if p.is_file()), None)
if ARCHIVO is None:
    raise FileNotFoundError("Sube corpus_auditoria_v2.csv al panel Archivos o conserva data/ en el paquete extraído.")
raw = pd.read_csv(ARCHIVO, encoding="utf-8", keep_default_na=False,
                  na_values=["__NULO__"], dtype="string")
assert set(raw.columns) == {"id", "text", "label", "family"}
HUELLA_ORIGINAL = hashlib.sha256(ARCHIVO.read_bytes()).hexdigest()
OUT = Path.cwd() / "entrega"
OUT.mkdir(exist_ok=True)
completados = set()
print("Carga lista. Ejecuta tu auditoría antes de transformar.")


Carga lista. Ejecuta tu auditoría antes de transformar.


## 1. Auditar antes de transformar

Define en 2–3 frases la entrada, la salida que entregarás y dos condiciones para considerar confiable el resultado. Carga el archivo sin modificarlo y documenta el diccionario de las cuatro columnas.

Calcula filas, columnas, IDs repetidos, distribución de etiquetas incluyendo ausencias, número de familias y faltantes por columna. Distingue nulo, cadena vacía y solo espacios. Cuenta repeticiones exactas de text indicando si cuentas filas repetidas después de la primera o todas las filas involucradas.

Lee ejemplos: distingue duplicado exacto, variante relacionada y etiqueta discutible. Muestra al menos un caso que requiera revisión de etiqueta o de información insuficiente y una acción justificada. No infieras etiquetas a partir de family. Usa head, isna, duplicated, value_counts y groupby cuando respondan una pregunta concreta; no basta imprimir veinte comandos.

Antecedentes: Clase 1, práctica 1–2; Clase 2, pasos 1–3.

In [3]:
from collections import Counter
from IPython.display import display

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

dicc = pd.DataFrame({
    "tipo": raw.dtypes.astype(str),
    "nulos": raw.isna().sum(),
    "vacios": (raw == "").sum(),
    "solo_espacios": raw.apply(lambda c: c.str.fullmatch(r"\s+").fillna(False).sum()),
    "valores_distintos": raw.nunique(),
})
print("filas y columnas:", raw.shape)
print("IDs repetidos:", raw["id"].duplicated().sum())
print("familias:", raw["family"].nunique())
display(dicc)
display(raw["label"].replace("", "(cadena vacía)").value_counts(dropna=False).rename("filas"))

vistos = raw["text"].dropna()
irregulares = vistos[vistos.str.contains(r"^\s|\s$|\s{2,}")]
print("textos con espacios al inicio, al final o dobles:", len(irregulares))

con_texto = raw[raw["text"].notna()]
repetida = con_texto["text"].duplicated()
involucrada = con_texto["text"].duplicated(keep=False)
print("repetidas después de la primera:", repetida.sum())
print("filas involucradas en algún grupo repetido:", involucrada.sum())
display(con_texto.loc[involucrada, ["id", "family", "label", "text"]])

print("tamaños de familia (filas: familias):", raw.groupby("family").size().value_counts().sort_index().to_dict())
con_etiqueta = raw[raw["label"] != ""]
n_etiquetas = con_etiqueta.groupby("family")["label"].nunique()
print("familias con más de una etiqueta:", list(n_etiquetas[n_etiquetas > 1].index))
revisar = raw["family"].isin(["F001", "F029", "F030"]) | raw["id"].isin(["AUD-013", "AUD-021"])
display(raw.loc[revisar, ["id", "family", "label", "text"]])

completados.add(1)

filas y columnas: (97, 4)
IDs repetidos: 0
familias: 32


,tipo,nulos,vacios,solo_espacios,valores_distintos
id,string,0,0,0,97
text,string,1,1,1,94
label,string,0,1,0,3
family,string,0,0,0,32


label
acceso            49
pagos             47
(cadena vacía)     1
Name: filas, dtype: int64[pyarrow]

textos con espacios al inicio, al final o dobles: 10
repetidas después de la primera: 2
filas involucradas en algún grupo repetido: 4


,id,family,label,text
4,AUD-005,F002,acceso,"Buen día, el código de verificación no llega a mi correo. ¿Me pueden orientar?"
72,AUD-073,F002,acceso,"Buen día, el código de verificación no llega a mi correo. ¿Me pueden orientar?"
88,AUD-089,F030,pagos,El código de verificación no llega y no puedo entrar.
89,AUD-090,F030,acceso,El código de verificación no llega y no puedo entrar.


tamaños de familia (filas: familias): {3: 31, 4: 1}
familias con más de una etiqueta: ['F030']


,id,family,label,text
0,AUD-001,F001,acceso,Olvidé la contraseña y necesito restablecerla.
1,AUD-002,F001,acceso,"Buen día, olvidé la contraseña y necesito restablecerla. ¿Me pueden orientar?"
2,AUD-003,F001,acceso,"Olvidé la contraseña y necesito restablecerla; ya reinicié y sigue igual, urge tantito."
12,AUD-013,F005,acceso,
20,AUD-021,F007,,"El portal rechaza mi nombre de usuario; ya reinicié y sigue igual, urge tantito."
85,AUD-086,F029,acceso,<NA>
86,AUD-087,F029,acceso,
87,AUD-088,F029,acceso,Ayuda.
88,AUD-089,F030,pagos,El código de verificación no llega y no puedo entrar.
89,AUD-090,F030,acceso,El código de verificación no llega y no puedo entrar.


### Análisis

Entrada: corpus_auditoria_v2.csv (97 mensajes de soporte). Salida: corpus_preparado.csv, un registro por ID y tres particiones por familia. Será confiable si cada ID queda explicado en el registro y ninguna familia ni texto equivalente cruza particiones.

Hay 97 filas, 4 columnas, 0 IDs repetidos y 32 familias. En text hay un nulo (AUD-086), una cadena vacía (AUD-013) y un texto de solo espacios (AUD-087); en label, una cadena vacía (AUD-021). Cuento 2 repeticiones exactas después de la primera (4 filas involucradas). AUD-073 repite a AUD-005 con la misma etiqueta, pero AUD-089 y AUD-090 son el mismo texto con etiquetas distintas (pagos y acceso). Esa familia (F030) va a cuarentena, igual que la F029, que no tiene texto útil. AUD-091 parece de acceso, pero sin evidencia no le cambio la etiqueta.

## 2. Demostrar Unicode y decidir una política

Encuentra en el corpus al menos un caso donde NFC cambie la representación interna. Compara dos representaciones pertinentes con repr, len o puntos de código antes y después de unicodedata.normalize("NFC", texto). Explica por qué pueden verse iguales y no comparar igual, qué cambia y qué no cambia. No basta usar solo el ejemplo «á» del profesor.

Escribe una tabla de decisiones que cubra: NFC y espacios; mayúsculas; acentos y ñ; puntuación; números y moneda; emojis; URLs y correos; stopwords y negación. Puedes agrupar operaciones relacionadas. En cada fila indica aplicar o conservar, motivo y una comprobación; una operación que decides no aplicar también es una decisión.

Define el orden de operaciones. No presupongas que mayor cantidad de limpieza es mejor. Compara duplicados con una clave explícita tras revisar NFC y espacios, sin confundir igualdad de texto con igualdad de casos.

Antecedentes: Clase 2, pasos 4–7.

In [4]:
def puntos(s):
    return " ".join(f"U+{ord(c):04X}" for c in s)

def nfc_espacios(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", s)).strip()

def clave_texto(s):
    return nfc_espacios(s).casefold()

def sin_acentos(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if not unicodedata.combining(c))

cambian = [i for i, s in zip(raw["id"], raw["text"]) if isinstance(s, str) and s != unicodedata.normalize("NFC", s)]
print("textos que NFC modifica:", cambian)

antes = raw.loc[raw["id"] == "AUD-081", "text"].iloc[0]
vecino = raw.loc[raw["id"] == "AUD-080", "text"].iloc[0]
despues = unicodedata.normalize("NFC", antes)
print("AUD-081 antes  :", repr(antes), len(antes))
print("AUD-081 después:", repr(despues), len(despues))
for i, c in enumerate(antes):
    if unicodedata.combining(c):
        print("antes:", repr(antes[i - 1:i + 1]), puntos(antes[i - 1:i + 1]))
for c in despues:
    if c in "óñ":
        print("después:", repr(c), puntos(c))
print("AUD-081 == AUD-080 antes:", antes == vecino, "| después:", despues == vecino)

for i in ["AUD-092", "AUD-093", "AUD-094"]:
    t = raw.loc[raw["id"] == i, "text"].iloc[0]
    print(i, repr(t[:22]), "->", repr(sin_acentos(t)[:22]))
print("'Sí' y 'Si' tras quitar acentos:", sin_acentos("Sí") == sin_acentos("Si"))

textos = raw["text"].dropna()
pasos = {
    "texto crudo": textos,
    "NFC": textos.map(lambda s: unicodedata.normalize("NFC", s)),
    "NFC y espacios": textos.map(nfc_espacios),
    "clave (NFC, espacios, minúsculas)": textos.map(clave_texto),
}
filas = []
for nombre, s in pasos.items():
    s = s[s != ""]
    filas.append({"comparación": nombre, "repetidas tras la primera": s.duplicated().sum(),
                  "filas involucradas": s.duplicated(keep=False).sum()})
display(pd.DataFrame(filas))

claves = pasos["clave (NFC, espacios, minúsculas)"]
claves = claves[claves != ""]
grupo = claves[claves.duplicated(keep=False)]
display(raw.loc[grupo.index, ["id", "family", "label"]].assign(clave=grupo).sort_values("clave"))

completados.add(2)

textos que NFC modifica: ['AUD-081']
AUD-081 antes  : 'Necesito información para recuperar mi contraseña.' 52
AUD-081 después: 'Necesito información para recuperar mi contraseña.' 50
antes: 'ó' U+006F U+0301
antes: 'ñ' U+006E U+0303
después: 'ó' U+00F3
después: 'ñ' U+00F1
AUD-081 == AUD-080 antes: False | después: True
AUD-092 'Este año cambié de tel' -> 'Este ano cambie de tel'
AUD-093 'Sí, quiero mantener la' -> 'Si, quiero mantener la'
AUD-094 'Si cambio de teléfono,' -> 'Si cambio de telefono,'
'Sí' y 'Si' tras quitar acentos: True


,comparación,repetidas tras la primera,filas involucradas
0,texto crudo,2,4
1,NFC,3,6
2,NFC y espacios,3,6
3,"clave (NFC, espacios, minúsculas)",3,6


,id,family,label,clave
4,AUD-005,F002,acceso,"buen día, el código de verificación no llega a mi correo. ¿me pueden orientar?"
72,AUD-073,F002,acceso,"buen día, el código de verificación no llega a mi correo. ¿me pueden orientar?"
88,AUD-089,F030,pagos,el código de verificación no llega y no puedo entrar.
89,AUD-090,F030,acceso,el código de verificación no llega y no puedo entrar.
79,AUD-080,F027,acceso,necesito información para recuperar mi contraseña.
80,AUD-081,F027,acceso,necesito información para recuperar mi contraseña.


### Análisis

Solo AUD-081 cambia con NFC. Se ve igual que AUD-080, pero trae la ó como o + U+0301 y la ñ como n + U+0303 (52 caracteres, 50 después). Antes de normalizar no comparan igual y después sí: cambia la representación interna, no lo que se lee. Por eso NFC va antes de buscar duplicados (2 repetidas en crudo, 3 con la clave). Espacios y minúsculas no sumaron más aquí.

| Operación | Decisión | Motivo | Comprobación |
| --- | --- | --- | --- |
| NFC | aplicar | una sola forma por letra | AUD-081 == AUD-080 solo después |
| Espacios | aplicar (colapsar y recortar) | 10 textos con espacios de más | AUD-001 |
| Mayúsculas | conservar | el NO de AUD-075 marca énfasis; minúsculas solo en la clave de duplicados | AUD-075 igual |
| Acentos y ñ | conservar | sí/si, año/ano | sin_acentos junta AUD-093 y AUD-094 |
| Puntuación | conservar | ¿?, ¡! y ; marcan pregunta y tono | AUD-077 y AUD-095 |
| Números y moneda | conservar | $1,250.50 contra $1,520.50 es el problema del mensaje | AUD-077 igual |
| Emojis | conservar | los de AUD-095 y AUD-096 dicen el ánimo | AUD-095 igual |
| URLs y correos | aplicar (enmascarar) | datos personales posibles | AUD-083 y AUD-085 |
| Stopwords y negación | conservar | no, sí y si cambian el sentido | conteo de "no" igual en los 87 textos |

Orden: nulos y vacíos, NFC, espacios, clave de duplicados, URL y luego correo (los patrones cortan en espacio). AUD-089 y AUD-090 comparten clave con etiqueta distinta, así que son un conflicto y no un duplicado.

## 3. Preparar sin perder trazabilidad

Implementa tu política como función y aplícala a una copia. Conserva text_original sin alteraciones y crea text_processed. No edites el CSV a mano. Decide el tratamiento de ausencias, duplicados, etiquetas dudosas y textos insuficientes. Puedes poner registros en cuarentena (apartados para revisión, sin usarlos todavía) en vez de resolver una duda sin evidencia.

Lleva un registro con una fila por ID original: estado (conservar, cuarentena o duplicado), motivo y, cuando corresponda, ID del registro conservado equivalente. No elimines filas sin dejar rastro. No cambies una etiqueta por adivinación; una corrección necesita evidencia y política explícitas.

Identifica información potencialmente sensible y aplica o justifica una política para ella. Basta una regex sencilla de correos, URLs o espacios; puedes adaptar la de Clase 2. Explica sus límites. No tienes que desarrollar un detector universal.

Muestra una tabla antes/después de al menos seis filas pertinentes del corpus, incluyendo cambios y decisiones de conservar. Comprueba que sobreviven acentos, ñ y negación cuando afectan el sentido. Prueba también estos contrastes: «No quiero cancelar mi cuenta.»; «sí» frente a «si»; «Este año renové mi cuenta.». Añade un caso propio que pueda romper tu regla. No cambies la fuente para superar una comprobación.

Antecedentes: Clase 2, pasos 5–7 y 12.

In [5]:
PATRON_URL = re.compile(r"https?://\S*[^\s.,;:!?)]")
PATRON_CORREO = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")
ETIQUETAS = {"acceso", "pagos"}
MIN_PALABRAS = 3

def preparar(texto):
    if not isinstance(texto, str):
        raise TypeError("se esperaba texto")
    t = nfc_espacios(texto)
    t = PATRON_URL.sub("URL_OCULTA", t)
    return PATRON_CORREO.sub("CORREO_OCULTO", t)

def motivo_cuarentena(texto, etiqueta):
    if pd.isna(texto):
        return "texto no recibido (__NULO__)"
    t = nfc_espacios(texto)
    if t == "":
        return "texto vacío" if texto == "" else "texto de solo espacios"
    if len(t.split()) < MIN_PALABRAS:
        return "texto insuficiente (menos de 3 palabras)"
    if etiqueta not in ETIQUETAS:
        return "etiqueta ausente (cadena vacía); no se infiere de family ni del texto"
    return None

def construir_registro(tabla):
    filas = []
    for r in tabla.itertuples(index=False):
        m = motivo_cuarentena(r.text, r.label)
        filas.append({"id": r.id, "family": r.family, "text_original": r.text, "label": r.label,
                      "clave": None if m else clave_texto(r.text),
                      "estado": "cuarentena" if m else "conservar",
                      "motivo": m or "texto y etiqueta utilizables", "id_conservado": ""})
    reg = pd.DataFrame(filas)

    vivos = reg[reg["estado"] == "conservar"]
    por_familia = vivos.groupby("family")["label"].nunique()
    por_clave = vivos.groupby("clave")["label"].nunique()
    familias_mal = set(por_familia[por_familia > 1].index)
    familias_mal |= set(vivos.loc[vivos["clave"].isin(por_clave[por_clave > 1].index), "family"])
    marca = (reg["estado"] == "conservar") & reg["family"].isin(familias_mal)
    reg.loc[marca, "estado"] = "cuarentena"
    reg.loc[marca, "motivo"] = "familia con etiquetas contradictorias; sin evidencia para decidir cuál es la correcta"

    vivos = reg[reg["estado"] == "conservar"]
    primero = vivos.groupby("clave")["id"].transform("first")
    repetidos = primero[vivos["id"] != primero]
    texto_de = reg.set_index("id")["text_original"]
    for i, base in repetidos.items():
        reg.loc[i, "estado"] = "duplicado"
        reg.loc[i, "id_conservado"] = base
        exacto = reg.loc[i, "text_original"] == texto_de[base]
        reg.loc[i, "motivo"] = ("texto exacto ya conservado" if exacto
                                else "mismo texto tras NFC, espacios o mayúsculas; misma etiqueta")
    return reg.drop(columns="clave")

registro = construir_registro(raw)
prep = raw.rename(columns={"text": "text_original"}).copy()
prep["text_processed"] = pd.Series([preparar(s) if isinstance(s, str) else pd.NA for s in prep["text_original"]],
                                   dtype="string", index=prep.index)
corpus = prep[prep["id"].isin(registro.loc[registro["estado"] == "conservar", "id"])].copy()
corpus["clave"] = corpus["text_original"].map(clave_texto)

display(registro["estado"].value_counts().rename("filas"))
display(registro.loc[registro["estado"] != "conservar", ["id", "family", "estado", "motivo", "id_conservado"]])

for col in ["clave", "text_processed"]:
    print(col, "| máximo de familias por valor:", corpus.groupby(col)["family"].nunique().max())

ids = ["AUD-001", "AUD-081", "AUD-083", "AUD-085", "AUD-077", "AUD-075", "AUD-095", "AUD-097"]
fila = prep.merge(registro[["id", "estado"]], on="id").set_index("id").loc[ids]
display(pd.DataFrame({"estado": fila["estado"],
                      "antes": fila["text_original"].map(repr),
                      "después": fila["text_processed"].map(repr)}))

def senales(t):
    return Counter(re.findall(r"[áéíóúüñÁÉÍÓÚÜÑ¿¡]", t)), len(re.findall(r"\bno\b", t, flags=re.I))

assert prep["text_original"].equals(raw["text"])
assert all(senales(unicodedata.normalize("NFC", o)) == senales(p)
           for o, p in zip(corpus["text_original"], corpus["text_processed"]))
print("conservadas con 'no':", corpus["text_original"].str.contains(r"\b[Nn][Oo]\b").sum(),
      "| con acento o ñ:", corpus["text_original"].map(lambda s: unicodedata.normalize("NFC", s)).str.contains("[áéíóúüñÁÉÍÓÚÜÑ]").sum(),
      "| de", len(corpus))

casos = ["No quiero cancelar mi cuenta.", "sí", "si", "Este año renové mi cuenta."]
for c in casos:
    print(repr(c), "->", repr(preparar(c)))
assert preparar(casos[0]) == casos[0]
assert preparar("sí") != preparar("si")
assert preparar(casos[3]) == casos[3]

propio = "Escríbeme a ana (arroba) example.invalid o llámame al 55 1234 5678."
print(repr(propio), "->", repr(preparar(propio)))
print("el correo y el teléfono siguen visibles:", preparar(propio) == propio)

completados.add(3)

estado
conservar     87
cuarentena     8
duplicado      2
Name: filas, dtype: int64

,id,family,estado,motivo,id_conservado
12,AUD-013,F005,cuarentena,texto vacío,
20,AUD-021,F007,cuarentena,etiqueta ausente (cadena vacía); no se infiere de family ni del texto,
72,AUD-073,F002,duplicado,texto exacto ya conservado,AUD-005
80,AUD-081,F027,duplicado,"mismo texto tras NFC, espacios o mayúsculas; misma etiqueta",AUD-080
85,AUD-086,F029,cuarentena,texto no recibido (__NULO__),
86,AUD-087,F029,cuarentena,texto de solo espacios,
87,AUD-088,F029,cuarentena,texto insuficiente (menos de 3 palabras),
88,AUD-089,F030,cuarentena,familia con etiquetas contradictorias; sin evidencia para decidir cuál es la correcta,
89,AUD-090,F030,cuarentena,familia con etiquetas contradictorias; sin evidencia para decidir cuál es la correcta,
90,AUD-091,F030,cuarentena,familia con etiquetas contradictorias; sin evidencia para decidir cuál es la correcta,


clave | máximo de familias por valor: 1
text_processed | máximo de familias por valor: 1


,estado,antes,después
id,,,
AUD-001,conservar,' Olvidé la contraseña y necesito restablecerla. ','Olvidé la contraseña y necesito restablecerla.'
AUD-081,duplicado,'Necesito información para recuperar mi contraseña.','Necesito información para recuperar mi contraseña.'
AUD-083,conservar,'Envíen la factura a persona@example.invalid; el cargo sigue pendiente.','Envíen la factura a CORREO_OCULTO; el cargo sigue pendiente.'
AUD-085,conservar,'La factura en https://soporte.example.invalid/factura?id=4821 no abre.','La factura en URL_OCULTA no abre.'
AUD-077,conservar,"'Pagué $1,250.50 y la factura muestra $1,520.50. ¿Pueden revisarla?'","'Pagué $1,250.50 y la factura muestra $1,520.50. ¿Pueden revisarla?'"
AUD-075,conservar,'NO quiero cancelar mi cuenta; necesito recuperar el acceso.','NO quiero cancelar mi cuenta; necesito recuperar el acceso.'
AUD-095,conservar,'¡Otra vez un cobro doble! 😠 Necesito que revisen la factura.','¡Otra vez un cobro doble! 😠 Necesito que revisen la factura.'
AUD-097,conservar,"'No puedo ver mi facutra del mes, ¿me ayudan?'","'No puedo ver mi facutra del mes, ¿me ayudan?'"


conservadas con 'no': 22 | con acento o ñ: 73 | de 87
'No quiero cancelar mi cuenta.' -> 'No quiero cancelar mi cuenta.'
'sí' -> 'sí'
'si' -> 'si'
'Este año renové mi cuenta.' -> 'Este año renové mi cuenta.'
'Escríbeme a ana (arroba) example.invalid o llámame al 55 1234 5678.' -> 'Escríbeme a ana (arroba) example.invalid o llámame al 55 1234 5678.'
el correo y el teléfono siguen visibles: True


### Análisis

Apliqué preparar a una copia: 87 conservar, 8 cuarentena y 2 duplicados (AUD-073 con AUD-005, AUD-081 con AUD-080). La tabla muestra cambios en AUD-001, 081, 083 y 085 y filas que se quedan igual. Los 87 conservan acentos, ñ, signos y "no", y los tres contrastes salen bien. Dejé "facutra" (AUD-097) como llegó.

Mi caso propio ("ana (arroba) example.invalid" con un teléfono) pasa sin enmascarar: la regex solo ve @ y http(s), y tampoco cubre nombres, el pedido 4821 ni RFC. No cambié la regla para que pasara. Enmascarar no anonimiza, y text_original viaja en los mismos CSV, aceptable solo por ser datos sintéticos.

## 4. Interpretar tokenización

Selecciona dos textos del corpus con rasgos distintos (por ejemplo, puntuación y moneda, o un correo y un acento). Usa el tokenizador de español de spaCy visto en clase. Muestra sus tokens y compara al menos una de esas salidas con dividir por espacios.

Explica qué unidades aparecen, una diferencia relevante y por qué importaría en el caso de soporte. Elige ejemplos para analizar, no solo una llamada a la biblioteca. No hay que lematizar, hacer stemming ni entrenar nada.

Antecedentes: Clase 2, demostración del paso 8.

In [6]:
nlp = spacy.blank("es")

for i in ["AUD-077", "AUD-083"]:
    t = raw.loc[raw["id"] == i, "text"].iloc[0]
    print(i, repr(t))
    print("  spaCy:", [x.text for x in nlp(t)])
    print("  split:", t.split())
    print("  cantidad spaCy:", len(nlp(t)), "| cantidad split:", len(t.split()))

p = corpus.loc[corpus["id"] == "AUD-083", "text_processed"].iloc[0]
print("AUD-083 procesado:", [x.text for x in nlp(p)])
print("con <CORREO>:", [x.text for x in nlp("Envíen la factura a <CORREO>; el cargo sigue pendiente.")])

completados.add(4)

AUD-077 'Pagué $1,250.50 y la factura muestra $1,520.50. ¿Pueden revisarla?'
  spaCy: ['Pagué', '$', '1,250.50', 'y', 'la', 'factura', 'muestra', '$', '1,520.50', '.', '¿', 'Pueden', 'revisarla', '?']
  split: ['Pagué', '$1,250.50', 'y', 'la', 'factura', 'muestra', '$1,520.50.', '¿Pueden', 'revisarla?']
  cantidad spaCy: 14 | cantidad split: 9
AUD-083 'Envíen la factura a persona@example.invalid; el cargo sigue pendiente.'
  spaCy: ['Envíen', 'la', 'factura', 'a', 'persona@example.invalid', ';', 'el', 'cargo', 'sigue', 'pendiente', '.']
  split: ['Envíen', 'la', 'factura', 'a', 'persona@example.invalid;', 'el', 'cargo', 'sigue', 'pendiente.']
  cantidad spaCy: 11 | cantidad split: 9
AUD-083 procesado: ['Envíen', 'la', 'factura', 'a', 'CORREO_OCULTO', ';', 'el', 'cargo', 'sigue', 'pendiente', '.']
con <CORREO>: ['Envíen', 'la', 'factura', 'a', '<', 'CORREO', '>', ';', 'el', 'cargo', 'sigue', 'pendiente', '.']


### Análisis

En AUD-077 spaCy separa "$" de la cifra, el punto final y el ¿, y da 14 tokens contra 9 de split, que deja "$1,520.50." y "¿Pueden" pegados. En soporte importa porque el mismo importe aparecería escrito de dos formas. En AUD-083 spaCy deja el correo entero y aparta el ";", mientras que split lo entrega como "persona@example.invalid;". Con "<CORREO>" spaCy parte el marcador en tres tokens, así que usé CORREO_OCULTO.

## 5. Separar y verificar

Separa el corpus conservado en train, validation y test con una semilla declarada y proporción razonable; por ejemplo 70/15/15 u 80/10/10. Separa familias completas, no filas independientes. Explica el propósito de cada partición. Documenta las proporciones reales por filas y por familias y la presencia de categorías; el redondeo puede impedir porcentajes exactos.

Para construir el reparto con operaciones conocidas: prepara una tabla con una fila por familia, ordena sus identificadores, mézclala con sample(frac=1, random_state=tu_semilla), decide los tamaños enteros y asigna a cada grupo una partición. Después incorpora esa asignación a las filas mediante map. Si agrupas por categoría, primero resuelve o aparta familias con etiquetas contradictorias. No se evalúa conocer una API nueva de scikit-learn.

Comprueba: IDs únicos; cada familia en una sola partición; ninguna clave de texto equivalente elegida ni text_processed idéntico cruza particiones; tres particiones no vacías; cobertura de categorías o limitación justificada. Si la preparación crea colisiones nuevas entre familias, revisa la política, agrupa esos casos relacionados o apártalos de forma documentada antes de dividir. No pruebes semillas hasta ocultar un problema.

En una copia, introduce un solapamiento de familia y muestra que tu control lo detecta, sin modificar el corpus de entrega. Explica por qué el mismo caso en train y test produciría una evaluación demasiado optimista. No calcules rendimiento ni uses test para escoger reglas por desempeño.

Antecedentes: Clase 1, sample del paso 2; Clase 2, pasos 10–11. El procedimiento de asignación se proporciona aquí como apoyo; se evalúan la decisión de agrupar y su validación.

In [7]:
SEMILLA = 42
TAMANOS = {"acceso": (10, 3, 2), "pagos": (11, 2, 2)}

familias = (corpus.groupby("family").agg(label=("label", "first"), filas=("id", "size"))
            .reset_index().sort_values("family"))
asignacion = {}
for etiqueta, (n_train, n_val, n_test) in TAMANOS.items():
    g = familias[familias["label"] == etiqueta].sample(frac=1, random_state=SEMILLA)
    assert len(g) == n_train + n_val + n_test
    nombres = ["train"] * n_train + ["validation"] * n_val + ["test"] * n_test
    asignacion.update(dict(zip(g["family"], nombres)))
corpus["split"] = corpus["family"].map(asignacion)

def validar_particiones(tabla):
    problemas = []
    if tabla["id"].duplicated().any():
        problemas.append("IDs repetidos")
    por_familia = tabla.groupby("family")["split"].nunique()
    if (por_familia > 1).any():
        problemas.append(f"familias en más de una partición: {sorted(por_familia[por_familia > 1].index)}")
    for col in ["clave", "text_processed"]:
        por_texto = tabla.groupby(col)["split"].nunique()
        if (por_texto > 1).any():
            problemas.append(f"{col} repetido entre particiones: {len(por_texto[por_texto > 1])}")
    if set(tabla["split"]) != {"train", "validation", "test"}:
        problemas.append("falta alguna partición")
    if (tabla.groupby("split")["label"].nunique() < tabla["label"].nunique()).any():
        problemas.append("alguna partición no tiene las dos categorías")
    return problemas

print("problemas en el corpus de entrega:", validar_particiones(corpus))

resumen = corpus.groupby("split").agg(filas=("id", "size"), familias=("family", "nunique")).loc[["train", "validation", "test"]]
resumen["% filas"] = (100 * resumen["filas"] / resumen["filas"].sum()).round(1)
resumen["% familias"] = (100 * resumen["familias"] / resumen["familias"].sum()).round(1)
display(resumen.join(pd.crosstab(corpus["split"], corpus["label"])))

copia = corpus.copy()
movida = copia.index[copia["split"] == "train"][0]
copia.loc[movida, "split"] = "test"
print("familia movida a test:", copia.loc[movida, "family"])
print("fuga por familia:", validar_particiones(copia))

copia = corpus.copy()
clon = copia[copia["split"] == "train"].head(1).assign(id="AUD-999", split="test")
print("fuga por texto repetido:", validar_particiones(pd.concat([copia, clon], ignore_index=True)))
print("problemas en el corpus de entrega tras las pruebas:", validar_particiones(corpus))

completados.add(5)

problemas en el corpus de entrega: []


,filas,familias,% filas,% familias,acceso,pagos
split,,,,,,
train,61,21,70.1,70.0,28,33
validation,15,5,17.2,16.7,9,6
test,11,4,12.6,13.3,5,6


familia movida a test: F001
fuga por familia: ["familias en más de una partición: ['F001']"]
fuga por texto repetido: ["familias en más de una partición: ['F001']", 'clave repetido entre particiones: 1', 'text_processed repetido entre particiones: 1']
problemas en el corpus de entrega tras las pruebas: []


### Análisis

Semilla 42. Repartí por familia, estratificando por etiqueta (15 familias de cada una, sin F029 ni F030): 10/3/2 en acceso y 11/2/2 en pagos. Quedan 61, 15 y 11 filas (70.1, 17.2 y 12.6%); no cuadra exacto porque cada familia va completa. Las dos categorías están en las tres particiones. Train es para ajustar, validation para comparar decisiones y test para la evaluación final.

Al mover una fila de F001 a test en una copia, la validación avisa la familia compartida; al copiar una fila de train con otro ID avisa también clave y text_processed. Con F001 repartida, test traería variantes de mensajes ya vistos y la evaluación saldría más optimista.

## 6. Exportar, reconstruir y explicar

Guarda corpus_preparado.csv con id, text_original, text_processed, label, family y split, en UTF-8. Guarda registro_decisiones.csv para todos los IDs iniciales, con estado y motivo. Los originales nulos pueden exportarse con el marcador `__NULO__`; documenta la misma convención al recargar.

Exporta train.csv, validation.csv y test.csv automáticamente desde la columna split. Son los archivos de partición pedidos por el plan; no los edites por separado. Añade manifiesto.json con versiones de Python, pandas y spaCy, semilla, política y nombre del archivo fuente. El diccionario de nuevas columnas y el reporte de decisiones van dentro del notebook.

Reabre tus CSV y verifica IDs, originales, particiones y conteos. Reinicia el kernel y ejecuta todo sin depender de celdas antiguas ni rutas de tu computadora. Conserva salidas e interpretación y descarga tus archivos si usas Colab.

Responde con evidencia de tu corpus: 1) ¿Por qué conservar o descartar «no» cambia la tarea? 2) ¿Qué se pierde al eliminar acentos y ñ? 3) ¿Por qué conservar el original? 4) ¿Por qué revisar NFC antes de declarar duplicados? 5) ¿Qué riesgo existe si una familia aparece en train y test? 6) ¿Qué limita tu tratamiento de datos sensibles?

Antecedentes: Clase 2, pasos 10–13.

In [8]:
COLUMNAS = ["id", "text_original", "text_processed", "label", "family", "split"]
PARTICIONES = ["train", "validation", "test"]
csv_args = dict(index=False, encoding="utf-8", lineterminator="\n", na_rep="__NULO__")

corpus[COLUMNAS].to_csv(OUT / "corpus_preparado.csv", **csv_args)
registro[["id", "family", "text_original", "estado", "motivo", "id_conservado"]].to_csv(
    OUT / "registro_decisiones.csv", **csv_args)
for nombre in PARTICIONES:
    corpus.loc[corpus["split"] == nombre, COLUMNAS].to_csv(OUT / f"{nombre}.csv", **csv_args)

manifiesto = {
    "archivo_fuente": ARCHIVO.name,
    "sha256_fuente": HUELLA_ORIGINAL,
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "spacy": spacy.__version__,
    "tokenizador": 'spacy.blank("es")',
    "semilla": SEMILLA,
    "reparto_familias": {k: list(v) for k, v in TAMANOS.items()},
    "politica": {
        "aplicar": ["NFC", "colapsar espacios y quitar los de los bordes",
                    "URL a URL_OCULTA", "correo a CORREO_OCULTO"],
        "conservar": ["mayúsculas", "acentos y ñ", "puntuación", "números y moneda", "emojis",
                      "stopwords y negación"],
        "clave_duplicados": "NFC, espacios y minúsculas sobre text_original; misma etiqueta",
        "cuarentena": ["texto nulo, vacío o de solo espacios", "menos de 3 palabras",
                       "etiqueta ausente", "familia con etiquetas contradictorias"],
        "nulos": "se exportan como __NULO__; al recargar usar keep_default_na=False y na_values=['__NULO__']",
    },
}
(OUT / "manifiesto.json").write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding="utf-8")

display(pd.DataFrame([
    ("text_original", "texto tal como llegó; __NULO__ si no se recibió"),
    ("text_processed", "text_original tras NFC, espacios y enmascarado de correos y URLs"),
    ("split", "train, validation o test; asignado por familia"),
    ("estado", "conservar, cuarentena o duplicado (registro_decisiones.csv)"),
    ("motivo", "razón de la decisión sobre ese ID"),
    ("id_conservado", "para duplicados, el ID del registro que se quedó"),
], columns=["columna", "contenido"]))
display(registro.groupby(["estado", "motivo"]).size().rename("filas").reset_index())

carga = dict(encoding="utf-8", keep_default_na=False, na_values=["__NULO__"], dtype="string")
c2 = pd.read_csv(OUT / "corpus_preparado.csv", **carga)
r2 = pd.read_csv(OUT / "registro_decisiones.csv", **carga)
p2 = {n: pd.read_csv(OUT / f"{n}.csv", **carga) for n in PARTICIONES}
original = raw.set_index("id")["text"]

assert r2["id"].is_unique and list(r2["id"]) == list(raw["id"])
assert r2.set_index("id")["text_original"].equals(original)
assert c2.set_index("id")["text_original"].equals(original.loc[c2["id"]])
assert set(c2["id"]) == set(r2.loc[r2["estado"] == "conservar", "id"])
assert sum(len(t) for t in p2.values()) == len(c2)
assert all((c2.loc[c2["split"] == n, "id"].tolist() == p2[n]["id"].tolist()) for n in PARTICIONES)
assert [preparar(t) for t in c2["text_original"]] == c2["text_processed"].tolist()
c2["clave"] = c2["text_original"].map(clave_texto)
assert validar_particiones(c2) == []
assert json.loads((OUT / "manifiesto.json").read_text(encoding="utf-8"))["semilla"] == SEMILLA

print("estados en el registro:", r2["estado"].value_counts().to_dict())
print("filas por partición:", {n: len(t) for n, t in p2.items()})
print("archivos en entrega/:", sorted(p.name for p in OUT.iterdir()))

completados.add(6)

,columna,contenido
0,text_original,texto tal como llegó; __NULO__ si no se recibió
1,text_processed,"text_original tras NFC, espacios y enmascarado de correos y URLs"
2,split,"train, validation o test; asignado por familia"
3,estado,"conservar, cuarentena o duplicado (registro_decisiones.csv)"
4,motivo,razón de la decisión sobre ese ID
5,id_conservado,"para duplicados, el ID del registro que se quedó"


,estado,motivo,filas
0,conservar,texto y etiqueta utilizables,87
1,cuarentena,etiqueta ausente (cadena vacía); no se infiere de family ni del texto,1
2,cuarentena,familia con etiquetas contradictorias; sin evidencia para decidir cuál es la correcta,3
3,cuarentena,texto de solo espacios,1
4,cuarentena,texto insuficiente (menos de 3 palabras),1
5,cuarentena,texto no recibido (__NULO__),1
6,cuarentena,texto vacío,1
7,duplicado,"mismo texto tras NFC, espacios o mayúsculas; misma etiqueta",1
8,duplicado,texto exacto ya conservado,1


estados en el registro: {'conservar': 87, 'cuarentena': 8, 'duplicado': 2}
filas por partición: {'train': 61, 'validation': 15, 'test': 11}
archivos en entrega/: ['corpus_preparado.csv', 'manifiesto.json', 'registro_decisiones.csv', 'test.csv', 'train.csv', 'validation.csv']


### Análisis

1) Sin "no", AUD-074 queda como "quiero cancelar mi cuenta". 22 de los 87 textos lo usan.

2) 73 de 87 textos llevan acento o ñ. Sin ellos AUD-093 ("Sí") y AUD-094 ("Si") quedan iguales y el "año" de AUD-092 pasa a "ano".

3) text_processed se regenera desde el original y al revés no. Si la política cambia, se parte del original.

4) Hasta normalizar, AUD-080 y AUD-081 se cuentan como dos casos y son uno.

5) La evaluación se infla, como en el apartado 5.

6) La regex solo ve correos con @ y URLs con http(s). Nombres, teléfonos, pedidos y RFC pasan, y el original va en los mismos CSV.

## Una sola entrega
Entrega un ZIP con el notebook completado y ejecutado y la carpeta entrega/ que genera tu código. Esa carpeta contiene corpus_preparado.csv, registro_decisiones.csv, train.csv, validation.csv, test.csv y manifiesto.json. No se exige un reporte o README adicional: decisiones, diccionario ampliado, análisis, límites y orden de ejecución quedan en el notebook.

Escribe explicaciones breves, aproximadamente 350–550 palabras en total además de tablas y código; es una orientación, no un límite de calificación. Declara integrantes, aportaciones y ayuda recibida. No entregues capturas como sustituto del código. No incluyas paquetes instalados ni archivos del profesor.

## Criterios de aceptación y checklist
- [ ] Auditoría con conteos verificables y casos examinados.
- [ ] Problema, decisión, justificación y comprobación conectados.
- [ ] Caso NFC del corpus demostrado; acentos y ñ distinguidos de su representación.
- [ ] Textos originales e IDs conservados; exclusiones y duplicados trazables.
- [ ] Seis comparaciones antes/después y pruebas de español y negación.
- [ ] Dos textos tokenizados e interpretación de la comparación.
- [ ] Familias y textos equivalentes sin solapamientos; fallo controlado detectado.
- [ ] CSV reabiertos y notebook ejecutado desde un kernel nuevo.
- [ ] Privacidad, limitaciones, versiones y semilla documentadas.
- [ ] ZIP con los archivos indicados y sin información personal real.

Una ejecución de la plantilla vacía solo comprueba preparación y carga; no equivale a resolver la actividad. La celda final avisa si todavía quedan apartados pendientes.

## Rúbrica pública: 100 puntos internos

Se conservan los cinco criterios y pesos del plan de evaluación definitivo. Las nuevas precisiones explicitan la evidencia, sin aumentar el valor de la actividad. Niveles: sobresaliente 100% del criterio; competente 75%; parcial 50%; insuficiente 0%. La contribución a la materia es puntos internos × 0.08.

### Auditoría y procedencia - 20 puntos

- Sobresaliente: Identifica procedencia y columnas; distingue nulo, vacío y espacios; cuenta duplicados indicando el denominador y revisa familias, etiquetas y ejemplos verificables.

- Competente: Cubre estructura, faltantes y duplicados correctamente; omite un detalle secundario de procedencia o del conteo sin cambiar el diagnóstico central.

- Parcial: Aporta conteos reproducibles, pero confunde tipos de ausencia o duplicado, o deja una categoría importante de problemas sin examinar.

- Insuficiente: No presenta auditoría reproducible o sus afirmaciones contradicen los datos sin comprobación.

### Preservación lingüística - 25 puntos

- Sobresaliente: Explicita operaciones aplicadas y evitadas; demuestra NFC con el corpus, conserva original y sentido en español, prueba negación/acentos/ñ y explica dos tokenizaciones.

- Competente: La política y el caso NFC son correctos y conservan señales clave; falta una comparación secundaria o explicar una diferencia de tokenización.

- Parcial: Aplica una preparación parcialmente válida, pero no prueba NFC sobre el corpus o destruye señales de español sin justificar su impacto.

- Insuficiente: No implementa política verificable o elimina sistemáticamente señales esenciales sin evidencia de comprensión.

### Etiquetas y particiones - 25 puntos

- Sobresaliente: Documenta dudas de etiqueta sin adivinar; separa grupos completos con semilla y proporciones razonadas, verifica IDs/textos/familias/cobertura y detecta la fuga introducida.

- Competente: El reparto y las decisiones de etiqueta son válidos, con controles centrales sin fuga; falta un detalle secundario de proporciones o cobertura.

- Parcial: Existe reparto reproducible, pero cruza familias o textos, no detecta la contaminación, o resuelve etiquetas sin evidencia.

- Insuficiente: No entrega particiones comprobables o no existe un mecanismo de separación y verificación.

### Pruebas y reconstrucción - 20 puntos

- Sobresaliente: Reconstruye desde kernel nuevo con funciones; registra cada ID y motivo; exporta y recarga comprobando originales, conteos y particiones; documenta versiones y política.

- Competente: La reconstrucción y los archivos centrales funcionan; falta un dato secundario del manifiesto o una comprobación no esencial.

- Parcial: Solo una parte es reproducible, hay cambios manuales no reconstruidos, se pierde el original o la recarga no comprueba lo declarado.

- Insuficiente: No se puede reconstruir ningún resultado central con los archivos entregados.

### Argumentación y límites - 10 puntos

- Sobresaliente: Conecta problemas, decisiones y pruebas con ejemplos antes/después; justifica privacidad y orden; reconoce alternativas y limitaciones sin atribuir rendimiento no medido.

- Competente: Las decisiones centrales están justificadas y ligadas a evidencia; un límite o alternativa queda poco desarrollado.

- Parcial: Ofrece razones genéricas sin conectar varias decisiones con ejemplos, o confunde reemplazo de un campo con anonimización total.

- Insuficiente: No interpreta las salidas ni ofrece justificaciones verificables.

Se asigna el nivel más alto respaldado por la evidencia del criterio. Un error crítico no implica cero global ni una resta adicional.

## Integrantes y ayuda recibida

Entrega individual de Oscar Cárdenas Valdez (A01741965). Usé Claude como apoyo para armar el código y redactar las explicaciones. El notebook se ejecutó completo desde un kernel nuevo.

In [9]:
pendientes = sorted({1, 2, 3, 4, 5, 6} - completados)
assert not pendientes, f"faltan apartados: {pendientes}"

assert raw.shape == (97, 4) and raw["family"].nunique() == 32
assert len(corpus) == 87 and corpus["family"].nunique() == 30
assert registro["estado"].value_counts().to_dict() == {"conservar": 87, "cuarentena": 8, "duplicado": 2}
assert (len(antes), len(despues)) == (52, 50) and cambian == ["AUD-081"]
assert corpus["text_original"].str.contains(r"\b[Nn][Oo]\b").sum() == 22
assert corpus["text_original"].map(lambda s: unicodedata.normalize("NFC", s)).str.contains("[áéíóúüñÁÉÍÓÚÜÑ]").sum() == 73
assert resumen["filas"].to_dict() == {'train': 61, 'validation': 15, 'test': 11}
print("apartados completos:", sorted(completados))

apartados completos: [1, 2, 3, 4, 5, 6]
